# connections-rl — GRPO seed replicate (one run per session)

Set `SCALE` and `SEED` in cell 1, then **Save & Run All (batch)**. Walk away.

Four runs total to complete the multi-seed evidence:

| run | SCALE | SEED | approx |
|---|---|---|---|
| 1 | `'1.5b'` | 1 | ~10h |
| 2 | `'1.5b'` | 2 | ~10h |
| 3 | `'7b'` | 1 | ~6h |
| 4 | `'7b'` | 2 | ~6h |

~32 GPU-hours total against a 30/week quota, so budget two weeks. Seed 0 is
the already-published run; these add seeds 1 and 2 per scale.

**Design note:** only GRPO is re-seeded. All seeds warm-start from the *same*
published SFT adapter, which isolates GRPO run-to-run variance (the quantity
the single-seed critique targets) rather than confounding it with SFT variance.

Resumable: checkpoints sync to a per-seed Hub repo, so a 12h timeout just means
re-running this notebook with the same SCALE/SEED.

In [ ]:
# Cell 1 — SET THESE, then Save & Run All
SCALE = '7b'   # '1.5b' or '7b'
SEED = 1       # 1 or 2

import os
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
os.environ['HF_TOKEN'] = secrets.get_secret('HF_TOKEN')
os.environ['WANDB_API_KEY'] = secrets.get_secret('WANDB_API_KEY')
HF_USER = 'jacksonlukas'

assert SCALE in ('1.5b', '7b') and SEED in (1, 2)
CFG = 'configs/train/grpo.yaml' if SCALE == '1.5b' else 'configs/train/grpo-7b.yaml'
SFT_REPO = f"{HF_USER}/connections-rl-sft" + ('' if SCALE == '1.5b' else '-7b')
SFT_DIR = 'artifacts/sft' + ('' if SCALE == '1.5b' else '-7b')
TAG = f"{SCALE}-seed{SEED}"
OUT_DIR = f'artifacts/grpo-{TAG}'
CKPT_REPO = f'connections-rl-grpo-{TAG}-ckpt'
FINAL_REPO = f'{HF_USER}/connections-rl-grpo-{TAG}'
print(f'run: scale={SCALE} seed={SEED} -> {FINAL_REPO}')
!nvidia-smi | head -12

In [ ]:
# Cell 2 — repo, deps, data, SFT warm start (shared across seeds)
!git clone https://github.com/jacksonmlukas/connections-rl.git
%cd connections-rl
!pip install -q -e ".[train]" bitsandbytes
!pip uninstall -q -y torchao
!git clone --depth 1 https://github.com/jacksonmlukas/gvc-local.git /kaggle/working/gvc-local
os.environ['CONNECTIONS_PUZZLES'] = '/kaggle/working/gvc-local/data/puzzles/tagged_connections.json'
!python -m connections_rl.data.build --out data/splits

from huggingface_hub import snapshot_download
snapshot_download(SFT_REPO, local_dir=SFT_DIR, token=os.environ['HF_TOKEN'])
assert os.path.exists(f'{SFT_DIR}/adapter_config.json'), 'SFT adapter missing'
print('SFT warm start ready:', SFT_DIR)

In [ ]:
# Cell 3 — GRPO with seed override (auto-resumes from this seed's Hub checkpoints)
import subprocess
subprocess.run(['python', '-m', 'connections_rl.train.grpo',
                '--config', CFG,
                '--seed', str(SEED),
                '--output-dir', OUT_DIR,
                '--ckpt-hub-repo', CKPT_REPO,
                '--run-name', f'connections-rl-grpo-{TAG}'], check=True)

In [ ]:
# Cell 4 — push the seed's final adapter
from huggingface_hub import HfApi
assert os.path.exists(f'{OUT_DIR}/adapter_config.json'), 'GRPO produced no adapter'
api = HfApi()
api.create_repo(FINAL_REPO, exist_ok=True)
api.upload_folder(folder_path=OUT_DIR, repo_id=FINAL_REPO, ignore_patterns=['checkpoint-*'])
print('pushed', FINAL_REPO)